In [ ]:
# Coisas a Melhorar: 

# 1) Inserir função de modificação rápida de parâmetros

# 2) Atualizar as informações faltantes do projeto

## Lib import

In [ ]:
import datetime

import numpy as np
import matplotlib.pyplot as plt

from rocketpy import Environment, Flight, Function, MonteCarlo, Rocket, SolidMotor

from rocketpy.stochastic import (
    StochasticEnvironment,
    StochasticFlight,
    StochasticNoseCone,
    StochasticRailButtons,
    StochasticRocket,
    StochasticSolidMotor,
    StochasticTail,
    StochasticTrapezoidalFins,
)

## Launch Site

### Parte do código onde é definido as váriaveis do local e ambiente de lançamento

In [ ]:
env = Environment(
    latitude = -21.9419,                # Latitude da LASC
    longitude= -48.9531,                # Longitude da LASC
    timezone = 'America/Sao_Paulo',     # Fuso horário
    datum="WGS84"                       # Fonte das coordenadas (Padrão mundial do GPS)
)

DATETIME = datetime.datetime.fromisoformat("2026-02-06 12:00:00")       # Horário (Para verificar as condições do tempo)
env.set_date(DATETIME)

env.set_atmospheric_model(type="Ensemble", file="GEFS")                 # Modelo atmosférico (Ensemble é a melhor escolha para Monte Carlo)


env.set_topographic_profile(type="NASADEM_HGT", file="NASADEM_NC_s22w049.nc", dictionary="netCDF4", crs=None)       # Modelo topográfico

elevation = env.get_elevation_from_topographic_profile(env.latitude, env.longitude)

env.set_elevation(elevation)


env.all_info()      # Visualizar todas as informações

### Elevação do solo coletada pela API da NASA

## Stockhastic Launch Site

### Definição do desvio nos parâmetros do ambiente de lançamento para a simulação de Monte Carlo

In [ ]:
stochastic_env = StochasticEnvironment(
    environment=env,
    
    ensemble_member=list(range(env.num_ensemble_members)),      # Lista de previsões prováveis para o local

    wind_velocity_x_factor=(1.0, 0.1),      # Fator multiplicativo para a velocidade do vento na direção x, desvio padrão
    wind_velocity_y_factor=(1.0, 0.1),      # Fator multiplicativo para a velocidade do vento na direção y, desvio padrão
)

stochastic_env.visualize_attributes()

## Atlas Motor

### Definição do motor do Atlas

In [ ]:
Proton = SolidMotor(
    thrust_source="", #Arquivo .eng do motor
    dry_mass = x / 1000,
    dry_inertia=(xe-x, xe-x, xe-x),
    nozzle_radius= x / 1000,
    grain_number= x,
    grain_density= x,
    grain_outer_radius= x / 1000,
    grain_initial_inner_radius= x / 1000,
    grain_initial_height= x / 1000,
    grain_separation= x / 1000,
    grains_center_of_mass_position= x / 1000,
    center_of_dry_mass_position= x / 1000,
    nozzle_position= x /1000,  # Posição do fim do Nozzle
    throat_radius= x / 1000,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

Proton.info()

#### Aguardando prop disponibilizar o arquivo do Open Motor

## Stockhastic Atlas Motor

### Definição dos desvios nos parâmetros do motor do Atlas para simulação de Monte Carlo

In [ ]:
stochastic_Motor = StochasticSolidMotor(
    solid_motor=Proton,
    burn_start_time=(0, 0.1, "binomial"), # Média, Desvio-Padrão, tipo de distribuição
    grains_center_of_mass_position= x,
    grain_density= x,
    grain_separation= x / 1000,
    grain_initial_height= x / 1000,
    grain_initial_inner_radius= x / 1000,
    grain_outer_radius= x / 1000,
    #total_impulse=(6500, 1000), 
    throat_radius= x / 1000,
    nozzle_radius= x / 1000,
    nozzle_position= x,
)
stochastic_Motor.visualize_attributes()

#### Perguntar pra prop quais são os valores mais realistas de desvio para cada parâmetro

## Rocket structure

### Definição de toda a parte estrutural do foguete Atlas

In [ ]:
Atlas = Rocket(
    radius= 152/1000,
    mass= 13896/1000,
    inertia=(x, x, x), # Tirar do CAD
    power_off_drag=".csv",
    power_on_drag=".csv",
    center_of_mass_without_motor= 1268/1000,
    coordinate_system_orientation="nose_to_tail",
)

Nosecone = Atlas.add_nose(
    length= 850/1000, kind="Von Karman", position=0
)

Fins = Atlas.add_trapezoidal_fins(
    n=4,
    root_chord = x,
    tip_chord = x,
    span = x,
    position = x,
    cant_angle = x,
    sweep_length = x,
)


Atlas.add_motor(Proton, position= 1883/1000) 


boattail = Atlas.add_tail(
    top_radius=152/1000, bottom_radius=129/1000, length=300/1000, position=2348/1000, name="Boattail Cônico",
)

rail_buttons = Atlas.set_rail_buttons(
    upper_button_position= 1540/1000,
    lower_button_position= 2320/1000,
    angular_position=x,
)

#### Se formos usar o Monte Carlo, não é possível colocar aletas com formato personalizado (por pontos)

## Stockhastic rocket structure

### Definição dos desvios nos parâmetros estruturais do Atlas para simulação de Monte Carlo

In [ ]:
stochastic_Atlas = StochasticRocket(
    rocket=Atlas,
    radius= x,
    mass=(x/1000, x, "normal"), # Média, Desvio-Padrão, tipo de distribuição
    inertia_11=(0.019, 0),
    inertia_22=x,
    inertia_33=x,
    center_of_mass_without_motor=x,
    power_off_drag=(0.9081 / 1.05, 0.033),  # Multiplier for rocket's drag curve. Usually has a mean value of 1 
                                            # and a uncertainty of 5% to 10%

    power_on_drag=(0.9081 / 1.05, 0.033),   # Multiplier for rocket's drag curve. Usually has a mean value of 1 
                                            # and a uncertainty of 5% to 10%
)

stochastic_Nosecone = StochasticNoseCone(
    nosecone=Nosecone,
    length=x,
)

stochastic_Fins = StochasticTrapezoidalFins(
    trapezoidal_fins=Fins,
    root_chord=x,
    tip_chord=x,
    span=x,
)

stochastic_boattail = StochasticTail(
    tail=boattail,
    top_radius=x,
    bottom_radius=x,
    length=x,
)

stochastic_rail_buttons = StochasticRailButtons(
    rail_buttons=rail_buttons, buttons_distance=x
)


stochastic_Atlas.add_motor(stochastic_Motor, position=x)
stochastic_Atlas.add_nose(stochastic_Nosecone, position=(0, 0.001))
stochastic_Atlas.add_trapezoidal_fins(stochastic_Fins, position=(x, "normal"))
stochastic_Atlas.add_tail(stochastic_boattail)
stochastic_Atlas.set_rail_buttons(stochastic_rail_buttons, lower_button_position=(x, "normal"))

stochastic_Atlas.visualize_attributes()
#stochastic_Fins.visualize_attributes()
#stochastic_coifa.visualize_attributes()

#### Ver com estruturas quais são os valores mais realistas de desvio para cada posição de componente

## Visual configuration

### Checagem visual de componentes do foguete definida no código 

In [ ]:
Proton.draw()

Fins.draw()

Atlas.draw()
#Atlas.draw(filename="Atlas.png")

## Flight Conditions

### Definição das condições de voo na base de lançamento

In [ ]:
flightStage = Flight(
    rocket=Atlas,
    environment=env,
    rail_length=x,  # meters
    inclination=80,  # degrees
    heading=x,
)

## Stockhastic flight conditions

### Definição dos desvios nas condições de voo na base de lançamento para simulação de Monte Carlo 

In [ ]:
stochastic_flight = StochasticFlight(
    flight=flightStage,
    rail_length=(x, x)
    inclination=(80, 1),  # ângulo=80, desvio-padrão=1
    heading=(x, 2),     # ângulo=140, desvio-padrão=2
)

stochastic_flight.visualize_attributes()

## Monte Carlo Simulation

In [ ]:
Simulation = MonteCarlo(
    filename="Resultados_Monte-Carlo",
    environment=stochastic_env,
    rocket=stochastic_Atlas,
    flight=stochastic_flight,
)

Simulation.simulate(
    number_of_simulations=500,
    append=False,
    include_function_data=False,
    parallel=True,
    n_workers=None,
)

## Results

### Valores das variáveis de voo (Comum)

In [ ]:
Simulation.prints.all()

flightStage.prints.maximum_values()

### Salvando os resultados em dicionário

In [ ]:
simulation_general_results = []

simulation_results = {
    "out_of_rail_time": [],
    "out_of_rail_velocity": [],
    "apogee_time": [],
    "apogee": [],
    "apogee_x": [],
    "apogee_y": [],
    "t_final": [],
    "x_impact": [],
    "y_impact": [],
    "impact_velocity": [],
    "initial_stability_margin": [],
    "out_of_rail_stability_margin": [],
    "max_mach_number": [],
    "frontal_surface_wind": [],
    "lateral_surface_wind": [],
    "index": [],
}

simulation_output_file = open(str(filename) + ".outputs.txt", "r+")

# Read each line of the file and convert to dict
for line in simulation_output_file:
    # Skip comments lines
    if line[0] != "{":
        continue
    # Eval results and store them
    flight_result = eval(line)
    simulation_general_results.append(flight_result)
    for parameter_key, parameter_value in flight_result.items():
        simulation_results[parameter_key].append(parameter_value)

# Close data file
simulation_output_file.close()

# Print number of flights simulated
N = len(simulation_general_results)

### Apogeu resultante (Monte Carlo)

In [ ]:
print(
    f"Apogee Altitude - Mean Value: {np.mean(simulation_results['apogee']):0.3f} m"
)
print(
    f"Apogee Altitude - Standard Deviation: {np.std(simulation_results['apogee']):0.3f} m"
)

plt.figure()
plt.hist(simulation_results["apogee"], bins=int(N**0.5))
plt.title("Apogee Altitude")
plt.xlabel("Altitude (m)")
plt.ylabel("Number of Occurences")
plt.show()

# ---------------------------------------------------

print(
    f"Apogee Time - Mean Value: {np.mean(simulation_results['apogee_time']):0.3f} s"
)
print(
    f"Apogee Time - Standard Deviation: {np.std(simulation_results['apogee_time']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["apogee_time"], bins=int(N**0.5))
plt.title("Apogee Time")
plt.xlabel("Time (s)")
plt.ylabel("Number of Occurences")
plt.show()

### Margem de estabilidade (Comum + Monte Carlo)

In [ ]:
Atlas.plots.static_margin()

# ---------------------------------------

print(
    f"Stability Margin - Mean Value: {np.mean(simulation_results['initial_stability_margin']):0.3f} s"
)
print(
    f"Stability Margin - Standard Deviation: {np.std(simulation_results['initial_stability_margin']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["initial_stability_margin"], bins=int(N**0.5))
plt.title("Stability Margin")
plt.xlabel("Cal")
plt.ylabel("Number of Occurences")
plt.show()

### Velocidades de vento / Velocidade de impacto no solo (Monte Carlo)

In [ ]:
print(
    f"Frontal Surface Wind - Mean Value: {np.mean(simulation_results['frontal_surface_wind']):0.3f} s"
)
print(
    f"Frontal Surface Wind - Standard Deviation: {np.std(simulation_results['frontal_surface_wind']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["frontal_surface_wind"], bins=int(N**0.5))
plt.title("Frontal Surface Wind")
plt.xlabel("Velocity (m/s)")
plt.ylabel("Number of Occurences")
plt.show()

# ----------------------------------------------------------------

print(
    f"Lateral Surface Wind - Mean Value: {np.mean(simulation_results['lateral_surface_wind']):0.3f} s"
)
print(
    f"Lateral Surface Wind - Standard Deviation: {np.std(simulation_results['lateral_surface_wind']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["lateral_surface_wind"], bins=int(N**0.5))
plt.title("Lateral Surface Wind")
plt.xlabel("Velocity (m/s)")
plt.ylabel("Number of Occurences")
plt.show()

# ----------------------------------------------------------------

print(
    f"Impact Velocity - Mean Value: {np.mean(simulation_results['impact_velocity']):0.3f} s"
)
print(
    f"Impact Velocity - Standard Deviation: {np.std(simulation_results['impact_velocity']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["impact_velocity"], bins=int(N**0.5))
plt.title("Impact Velocity")
plt.xlabel("Velocity (m/s)")
plt.ylabel("Number of Occurences")
plt.show()


### Velocidade (Comum), Mach Max e Tempo de voo (Monte Carlo)

In [ ]:
flightStage.speed.plot(0, flightStage.apogee_time)

# ----------------------------------------------------------------

print(
    f"Flight time - Mean Value: {np.mean(simulation_results['t_final']):0.3f} s"
)
print(
    f"Flight time - Standard Deviation: {np.std(simulation_results['t_final']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["t_final"], bins=int(N**0.5))
plt.title("Flight time")
plt.xlabel("Time (s)")
plt.ylabel("Number of Occurences")
plt.show()

# ----------------------------------------------------------------

print(
    f"Max Mach Number - Mean Value: {np.mean(simulation_results['max_mach_number']):0.3f} s"
)
print(
    f"Max Mach Number - Standard Deviation: {np.std(simulation_results['max_mach_number']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["max_mach_number"], bins=int(N**0.5))
plt.title("Max Mach Number")
plt.xlabel("Mach")
plt.ylabel("Number of Occurences")
plt.show()

### Raio de aterrisagem provável (Monte Carlo)

In [ ]:
Simulation.plots.ellipses(xlim=(-1000, 1000), ylim=(-1000, 1000)) # Visualização do raio de aterrisagem

#image="Launch Site 2km_x_2km.png" é inserido no argumento "ellipses" para colocar background

### Trajetória de voo 3D

In [ ]:
flightStage.plots.trajectory_3d()

#flightStage.plots.trajectory_3d(filename="Trajectory_Stage.png")

In [ ]:
flightStage.export_kml(
    file_name = "Atlas.kml",
    extrude = True,
    altitude_mode = "relativetoground",
)